# UniProt Fetch

![UniProt Fetch](https://proto-bio.github.io/proto-assets/images/tool/uniprot/hero.png)

This notebook demonstrates `run_uniprot_fetch`, which provides a single-call interface to the UniProt REST API for retrieving protein entries. It supports two modes: direct lookup by accession, which is the fastest path when you already know the UniProt ID, and name-based search with organism filtering, which is useful when starting from a gene name in the literature. The search mode ranks candidates by gene name match quality, review status, and optionally by the presence of linked PDB structures, making it well-suited for workflows that need both a sequence and a structural template.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("uniprot")
display_overview("uniprot")
display_docs_section("uniprot", "Background")

## Available tools

In [ ]:
display_available_tools("uniprot")

### `run_uniprot_fetch`

`run_uniprot_fetch` fetches a protein entry from UniProt by accession or by searching with a gene name and organism. When given a UniProt accession directly, it retrieves the full entry in a single request. When given a name and organism, it issues two ranked search queries and selects the best candidate based on exact gene name match, reviewed (Swiss-Prot) status, and optionally PDB cross-reference availability. The returned output includes the primary accession, protein sequence, gene names, review status, linked PDB structure IDs, and the complete raw JSON entry for advanced access.

In [ ]:
from proto_tools import (
    UniProtFetchConfig,
    UniProtFetchInput,
    run_uniprot_fetch,
)

In [ ]:
# Display docs
display_api_reference("uniprot", "input", "run_uniprot_fetch")

# Fetch TP53 directly by its well-known UniProt accession
inputs = UniProtFetchInput(uniprot_id="P04637")

In [ ]:
# Display config docs
display_api_reference("uniprot", "config", "run_uniprot_fetch")

# Use default config | see docs above for all fields
config = UniProtFetchConfig()

In [ ]:
# Run the fetch
output = run_uniprot_fetch(inputs, config)

In [ ]:
# Display output docs
display_api_reference("uniprot", "output", "run_uniprot_fetch")

# Inspect the retrieved TP53 entry
print(f"Accession: {output.accession}")
print(f"Entry type: {output.entry_type}")
print(f"Gene names: {output.gene_names}")
print(f"Sequence length: {output.length}")
print(f"Preview: {output.sequence[:60]}...")
print(f"PDB cross-refs: {output.pdb_crossrefs[:5]} ({len(output.pdb_crossrefs)} total)")
print(f"Source: {output.source_url}")

#### Searching by Gene Name and Organism

When you do not have a UniProt accession, you can search by gene name and organism. The tool queries the UniProt search API, evaluates the top candidates, and ranks them by gene name match quality and review status. This is useful when starting from a gene name in the literature and needing to retrieve the canonical protein sequence.

In [ ]:
# Search for lacI by gene name and organism
search_inputs = UniProtFetchInput(
    target_name="lacI",
    organism="Escherichia coli",
)

search_output = run_uniprot_fetch(search_inputs, config)

print(f"Accession: {search_output.accession}")
print(f"Gene names: {search_output.gene_names}")
print(f"Sequence length: {search_output.length}")
print(f"Preview: {search_output.sequence[:60]}...")

#### Searching with PDB Cross-Reference Preference

Setting `prefer_pdb_crossref=True` prioritizes entries that have linked PDB structures during the ranking step. This is valuable when you need both a protein sequence and an experimental structure for downstream tasks like inverse folding or structure-guided design. Here we search for human insulin and increase the candidate pool to improve recall.

In [ ]:
# Search for human insulin, preferring entries with PDB structures
pdb_inputs = UniProtFetchInput(
    target_name="insulin",
    organism="Homo sapiens",
    prefer_pdb_crossref=True,
    max_candidates=10,
)

pdb_output = run_uniprot_fetch(pdb_inputs, config)

print(f"Accession: {pdb_output.accession}")
print(f"Gene names: {pdb_output.gene_names}")
print(f"Sequence length: {pdb_output.length}")
if len(pdb_output.pdb_crossrefs) > 5:
    print(f"PDB cross-refs: {pdb_output.pdb_crossrefs[:5]}...")
else:
    print(f"PDB cross-refs: {pdb_output.pdb_crossrefs}")
print(f"Total PDB links: {len(pdb_output.pdb_crossrefs)}")

### `run_uniprot_search`

`run_uniprot_search` returns a ranked list of UniProtKB entries for free-text or UniProt query-syntax searches. Use it when you know what a protein does or where it comes from but not its gene symbol or accession: the tool reports each candidate's accession, names, organism and length so you can choose one, then pass the accession to `run_uniprot_fetch`. By default it keeps reviewed Swiss-Prot entries only.

In [ ]:
from proto_tools import (
    UniProtSearchConfig,
    UniProtSearchInput,
    run_uniprot_search,
)

In [ ]:
# Display docs
display_api_reference("uniprot", "input", "run_uniprot_search")

# Two descriptions, no gene symbols
search_queries = UniProtSearchInput(queries=["thermostable DNA polymerase Thermus aquaticus", "firefly luciferase"])

In [ ]:
# Display config docs
display_api_reference("uniprot", "config", "run_uniprot_search")

# Reviewed entries only (default), top 5 per query
search_config = UniProtSearchConfig(max_results=5)

In [ ]:
# Run the search
search_results = run_uniprot_search(search_queries, search_config)

In [ ]:
# Display output docs
display_api_reference("uniprot", "output", "run_uniprot_search")

# Several candidates per query: choose by organism and name rather than taking the first hit
for result in search_results.results:
    print(f"{result.query}  ({result.total_count} matches)")
    for hit in result.hits:
        print(f"  {hit.accession:8} {hit.entry_name:12} {hit.length:5} aa  {hit.organism}: {hit.protein_name}")

#### Filtering by Taxon

`taxon_id` restricts results to an NCBI taxonomy node and all of its descendants, which is useful for collecting orthologs. Here we collect reviewed p53 entries from mammals (taxon 40674).

In [ ]:
mammal_p53 = run_uniprot_search(
    UniProtSearchInput(queries="gene:TP53"),
    UniProtSearchConfig(taxon_id=40674, max_results=5),
)

for hit in mammal_p53.results[0].hits:
    print(f"{hit.accession:8} {hit.length:5} aa  {hit.organism}")

## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("uniprot_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'uniprot_results.json'}")